# 03a — Cloud Classification (Stage 1)

This notebook runs **Stage 1** of the attribution pipeline: classifying each record as cloud or non-cloud, and extracting platform mentions.

**Three steps:**
- **Step 1A — RegEx**: Fast pattern matching on `description` text (free, precise, limited recall)
- **Step 1B — LLM**: Anthropic API classification (comprehensive, contextual, ~$100 for full dataset)
- **Step 1C — Synthesis**: Combine RegEx + LLM with confidence scoring

> **Cost warning:** Step 1B calls the Anthropic API for every record (~216K calls). Estimated cost: ~$100 using Haiku 4.5. The LLM step includes crash recovery — results are checkpointed every 1,000 records and can be resumed if interrupted.

> **RegEx-only mode:** If no API key is set or LLM is skipped, the pipeline falls back to RegEx-only classification. All downstream analysis still works.

---

# Safe CSV loaders (avoid parser timeouts)
def safe_read_csv(path, usecols=None):
    return cls_mod.safe_read_csv(path, usecols=usecols)

MERGED_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '02_merged', 'merged_dataset.csv')
merged_df = safe_read_csv(MERGED_PATH)
print(f"Loaded {len(merged_df):,} records, ${merged_df['dollars'].sum()/1e9:.1f}B")
print(f"Record types: {merged_df['record_type'].value_counts().to_dict()}")

LLM_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified', 'llm_cache')
cached_results = cls_mod.load_llm_results(LLM_CACHE_DIR)
if cached_results is None or len(cached_results) == 0:
    print('No cached LLM results available (RegEx-only mode).')
else:
    print(f'Found cached LLM results: {len(cached_results):,} records')


In [ ]:
import pandas as pd
import numpy as np
import os, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

cls_mod = _import_module('cloud_classification',
    os.path.join(PROJECT_ROOT, 'notebooks', '03_multi-stage_attribution_pipeline', 'cloud_classification.py'))
    

In [ ]:
import pandas as pd
import numpy as np
import os, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

cls_mod = _import_module('cloud_classification',
    os.path.join(PROJECT_ROOT, 'notebooks', '03_multi-stage_attribution_pipeline', 'cloud_classification.py'))

## 1. Load Merged Dataset

In [ ]:
MERGED_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '02_merged', 'merged_dataset.csv')
merged_df = cls_mod.safe_read_csv(MERGED_PATH)
print(f'Loaded {len(merged_df):,} records, ${merged_df["dollars"].sum()/1e9:.1f}B')
print(f'Record types: {merged_df["record_type"].value_counts().to_dict()}')


## 2. Step 1A: RegEx Classification

We begin by scanning every record's `description` field against comprehensive 
regular expression patterns to identify cloud-related spending. This initial 
classification runs on the **full merged dataset** (216,604 records) and classifies 
primes and subcontracts independently.

Our RegEx patterns detect two categories of cloud spending:

1. **Generic cloud keywords**: Terms indicating cloud infrastructure or services 
   without specifying a particular vendor (e.g., "cloud computing", "cloud-based", 
   "IaaS", "PaaS", "SaaS", "serverless", "cloud migration")

2. **Platform-specific keywords**: Vendor names and proprietary services that 
   definitively indicate a specific cloud platform:
   - **AWS**: "AWS", "Amazon Web Services", platform-specific services (EC2, S3, 
     Lambda, SageMaker, DynamoDB, etc.), and custom chips (Inferentia, Trainium)
   - **Azure**: "Azure", "Microsoft Azure", platform services (Azure ML, Cosmos DB, 
     Azure Functions, etc.)
   - **Google Cloud**: "GCP", "Google Cloud", platform services (BigQuery, Vertex AI, 
     Cloud Functions, TPU, etc.)
   - **SaaS platforms**: Salesforce, ServiceNow, Workday, Oracle Cloud, IBM Cloud, etc.

The patterns include context-aware matching to avoid false positives (e.g., 
distinguishing AWS Lambda from "lambda calculus", AWS Aurora from "Aurora, Colorado").

This step produces three key outputs:
- `regex_is_cloud`: Boolean flag indicating cloud-related spending
- `regex_platforms`: List of detected platform(s) or "Unspecified" if generic 
  cloud terms only
- `regex_confidence`: Pattern match quality (for validation)

**Important**: RegEx classification is deterministic and fast (~18 seconds for 
216K records), making it ideal for initial filtering. However, it has known 
limitations in handling ambiguous descriptions, which we address through LLM 
classification in Step 1B.

In [ ]:
patterns = cls_mod.compile_patterns()
df = cls_mod.classify_cloud_regex(merged_df, patterns)

In [ ]:
# RegEx classification summary by record type
regex_summary = df.groupby('record_type').agg(
    total=('regex_is_cloud', 'count'),
    cloud=('regex_is_cloud', 'sum'),
    cloud_dollars=('dollars', lambda x: x[df.loc[x.index, 'regex_is_cloud']].sum())
)
regex_summary['cloud_rate'] = (regex_summary['cloud'] / regex_summary['total'] * 100).round(1)
print(regex_summary)

# Platform detection breakdown
print(f'\nRegEx platform detections (cloud records):')
cloud_regex = df[df['regex_is_cloud']]
for plat, count in cloud_regex['regex_platforms'].value_counts(dropna=False).items():
    label = str(plat) if pd.notna(plat) else 'No platform detected'
    dollars = cloud_regex.loc[cloud_regex['regex_platforms'] == plat, 'dollars'].sum() if pd.notna(plat) else cloud_regex.loc[cloud_regex['regex_platforms'].isna(), 'dollars'].sum()
    print(f'  {label:25s}: {count:>5,} records, ${dollars/1e6:>9.1f}M')

## 3. Step 1B: LLM Classification

Uses the Anthropic API (Haiku 4.5) to classify every record with a combined WHAT + WHO prompt:
- **Classification**: non-cloud / cloud-dependent / cloud-infrastructure
- **Platform**: AWS, Azure, Google Cloud, Oracle Cloud, IBM Cloud, Salesforce, Multi-cloud, Unspecified, N/A
- **Confidence**: high / medium / low
- **Evidence**: brief reasoning

**Crash recovery:** Results are checkpointed every 1,000 records. If interrupted, re-running this cell will resume from the last checkpoint.

**Cost:** ~$100 for the full dataset (~216K records) using Haiku 4.5.

In [ ]:
# Check for cached LLM results
LLM_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified', 'llm_cache')
cached_results = cls_mod.load_llm_results(LLM_CACHE_DIR)

if cached_results is not None:
    print(f'Found cached LLM results: {len(cached_results):,} records')
    print(f'Classification distribution:')
    print(cached_results['classification'].value_counts().to_string())
else:
    print('No cached LLM results found.')
    # Show cost estimate
    estimate = cls_mod.estimate_llm_cost(len(df))
    print(f'\nLLM classification would cost ~${estimate["estimated_cost_usd"]:.2f}')
    print(f'Estimated time: ~{estimate["estimated_minutes"]:.0f} minutes ({estimate["estimated_minutes"]/60:.1f} hours)')
    print(f'Model: {estimate["model"]}')

In [ ]:
# Run LLM classification (or load cached results)
# Set ANTHROPIC_API_KEY environment variable before running this cell
api_key = os.environ.get("ANTHROPIC_API_KEY")

# Ensure cached_results is loaded
LLM_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified', 'llm_cache')
if 'cached_results' not in globals() or cached_results is None:
    cached_results = cls_mod.load_llm_results(LLM_CACHE_DIR)

if cached_results is not None and len(cached_results) == len(df):
    print('Loading cached LLM results into dataframe...')
    df['llm_classification'] = cached_results['classification'].values
    df['llm_platform'] = cached_results['platform'].values
    df['llm_confidence'] = cached_results['confidence'].values
    df['llm_evidence'] = cached_results['evidence'].values
    print(f'Loaded {len(cached_results):,} LLM classifications.')
elif cached_results is not None:
    print('Cached LLM results length mismatch — refusing to attach.')
    print(f'cached_results: {len(cached_results):,} vs df: {len(df):,}')
elif api_key:
    print('Running LLM classification...')
    df = cls_mod.classify_cloud_llm(df, api_key, LLM_CACHE_DIR)
else:
    print('No API key found and no cached results. Skipping LLM classification.')
    print('Set ANTHROPIC_API_KEY to enable LLM classification.')
    print('Pipeline will proceed with RegEx-only classification.')


In [ ]:
# LLM classification summary (if available)
if 'llm_classification' in df.columns and df['llm_classification'].notna().any():
    print('LLM Classification Results:')
    print(f'\n  Classification distribution:')
    for cls, count in df['llm_classification'].value_counts().items():
        dollars = df.loc[df['llm_classification'] == cls, 'dollars'].sum()
        print(f'    {cls:25s}: {count:>8,} records, ${dollars/1e9:.2f}B')

    llm_cloud = df['llm_classification'].isin(['cloud-dependent', 'cloud-infrastructure'])
    print(f'\n  LLM cloud records: {llm_cloud.sum():,} / {len(df):,} ({llm_cloud.sum()/len(df)*100:.1f}%)')

    error_rate = (df['llm_classification'] == 'error').sum()
    print(f'  LLM errors: {error_rate:,} ({error_rate/len(df)*100:.2f}%)')

    print(f'\n  LLM confidence distribution:')
    for conf, count in df['llm_confidence'].value_counts().items():
        print(f'    {str(conf):10s}: {count:>8,}')

    print(f'\n  LLM platform detections (cloud records):')
    llm_plats = df.loc[llm_cloud, 'llm_platform'].value_counts()
    for plat, count in llm_plats.items():
        dollars = df.loc[llm_cloud & (df['llm_platform'] == plat), 'dollars'].sum()
        print(f'    {str(plat):25s}: {count:>6,} records, ${dollars/1e6:>9.1f}M')
else:
    print('LLM classification not available (RegEx-only mode).')

In [ ]:
# 1) Explicit hyperscaler mentions classified as non-cloud
if 'explicit_cloud_mention' in df.columns and 'llm_classification' in df.columns:
    mask = df['explicit_cloud_mention'] & ~df['llm_classification'].isin(['cloud-dependent', 'cloud-infrastructure'])
    subset = df[mask].copy()
    dollars = subset['dollars'].sum()
    print(f'Explicit hyperscaler mentions classified as non-cloud: {len(subset):,} records, ${dollars/1e9:.2f}B')
    if len(subset) > 0:
        cols = ['contractor_name', 'description', 'dollars', 'explicit_hyperscaler_mentions',
                'llm_classification', 'regex_is_cloud', 'regex_platforms']
        display(subset.sort_values('dollars', ascending=False)[cols].head(10))
else:
    print('explicit_hyperscaler_mentions not available in df')

# 2) Top-dollar non-cloud records (sample)
if 'cloud_classification' in df.columns:
    noncloud = df[df['cloud_classification'] == 'non-cloud'].copy()
else:
    noncloud = df[~df['is_cloud']].copy() if 'is_cloud' in df.columns else df.copy()

print(f'Top non-cloud sample (by dollars): {min(25, len(noncloud))} records')
cols = ['contractor_name', 'description', 'dollars', 'cloud_classification', 'llm_classification', 'regex_is_cloud']
cols = [c for c in cols if c in noncloud.columns]
display(noncloud.sort_values('dollars', ascending=False)[cols].head(25))


## 4. Step 1C: Synthesis

Combine RegEx and LLM results using confidence-weighted decision rules:

| Scenario | Decision | Confidence |
|----------|----------|------------|
| Both agree cloud + same platform | Cloud, that platform | HIGH |
| Both agree cloud, LLM has platform | Cloud, LLM platform | MEDIUM |
| LLM-only cloud (RegEx missed) | Cloud, LLM platform | MEDIUM |
| Both agree non-cloud | Non-cloud | HIGH |
| Conflict (RegEx cloud, LLM non-cloud) | Trust LLM → non-cloud | LOW |
| LLM error | Fall back to RegEx | LOW |

In [ ]:
df = cls_mod.synthesize_classification(df)

In [ ]:
# Detailed synthesis report
cloud_df = df[df['is_cloud']]
total = len(df)
cloud_n = len(cloud_df)
cloud_dollars = cloud_df['dollars'].sum()

print(f'Classification Summary:')
print(f'  Total records:   {total:>10,}')
print(f'  Cloud records:   {cloud_n:>10,}  ({cloud_n/total*100:.1f}%)')
print(f'  Cloud dollars:   ${cloud_dollars/1e9:>9.2f}B  ({cloud_dollars/df["dollars"].sum()*100:.1f}%)')

# 3-way classification breakdown
print(f'\nCloud Classification (3-way):')
for cls in ['non-cloud', 'cloud-dependent', 'cloud-infrastructure']:
    n = (df['cloud_classification'] == cls).sum()
    d = df.loc[df['cloud_classification'] == cls, 'dollars'].sum()
    print(f'  {cls:25s}: {n:>8,} records, ${d/1e9:.2f}B')

# Confidence distribution for cloud records
print(f'\nConfidence (cloud records):')
for conf, count in cloud_df['classification_confidence'].value_counts().items():
    dollars = cloud_df.loc[cloud_df['classification_confidence'] == conf, 'dollars'].sum()
    print(f'  {str(conf):15s}: {count:>6,} records, ${dollars/1e6:>9.1f}M')

# Classification source breakdown
print(f'\nClassification Source:')
for src, count in df['classification_source'].value_counts().items():
    print(f'  {str(src):35s}: {count:>8,}')

In [ ]:
# RegEx vs LLM agreement analysis (if both available)
if 'llm_classification' in df.columns and df['llm_classification'].notna().any():
    llm_is_cloud = df['llm_classification'].isin(['cloud-dependent', 'cloud-infrastructure'])
    regex_cloud = df['regex_is_cloud']

    both_cloud = (regex_cloud & llm_is_cloud).sum()
    both_noncloud = (~regex_cloud & ~llm_is_cloud).sum()
    llm_only = (~regex_cloud & llm_is_cloud).sum()
    regex_only = (regex_cloud & ~llm_is_cloud).sum()

    agree = both_cloud + both_noncloud
    total = len(df)

    print(f'RegEx vs LLM Agreement:')
    print(f'  Both cloud:     {both_cloud:>8,}  ({both_cloud/total*100:.2f}%)')
    print(f'  Both non-cloud: {both_noncloud:>8,}  ({both_noncloud/total*100:.2f}%)')
    print(f'  LLM-only cloud: {llm_only:>8,}  ({llm_only/total*100:.2f}%)')
    print(f'  RegEx-only cloud (conflict): {regex_only:>8,}  ({regex_only/total*100:.2f}%)')
    print(f'\n  Overall agreement: {agree:,} / {total:,} ({agree/total*100:.2f}%)')

    # Examine conflict records
    if regex_only > 0:
        print(f'\nConflict samples (RegEx=cloud, LLM=non-cloud):')
        conflicts = df[regex_cloud & ~llm_is_cloud].head(5)
        for _, row in conflicts.iterrows():
            desc = str(row['description'])[:80] if pd.notna(row['description']) else 'N/A'
            print(f'  Description: {desc}')
            print(f'    RegEx platforms: {row.get("regex_platforms", "N/A")}')
            print(f'    LLM class: {row.get("llm_classification", "N/A")}')
            print(f'    LLM evidence: {row.get("llm_evidence", "N/A")}')
            print()
else:
    print('LLM not available — skipping agreement analysis.')
    print('All records classified using RegEx only.')

## 5. Save Classified Dataset

In [ ]:
OUT_DIR = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified')
os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(OUT_DIR, 'classified_dataset.csv')
df.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'  Rows: {len(df):,}')
print(f'  Columns: {list(df.columns)}')
print(f'  Cloud records: {df["is_cloud"].sum():,}')
print(f'  Cloud dollars: ${df.loc[df["is_cloud"], "dollars"].sum()/1e9:.2f}B')

---

**Next:** [03b — Platform Attribution (Stage 2)](03b_platform_attribution.ipynb)